# LLM Prompting

There are lots of things that gets hidden from us when using a LLM chatbot, not talking just about the math that goes on but even simpler stuff like it being stateless, system prompts, temperature... I know a bit about those but not enough, so let's dive in.

## Statelessness
So when we call a model, it process the individual message we sent. The previous messages are not stored. Let me try it out.

In [1]:
from constants import MODEL
from litellm import completion

response = completion(
    model=MODEL,
    messages=[{"role": "user", "content": "Just wanted to let you know that my name is Rafael. Can you tell me in one line, what might be the origin of my name?"}]
)

In [5]:
response.choices[0].message.content

'Rafael is a name of Hebrew origin, derived from "Rapha\'el," meaning "God has healed," and is widely used in Spanish, Portuguese, and other Romance-speaking cultures.'

In [6]:
response_new = completion(
    model=MODEL,
    messages=[{"role": "user", "content": "Summarize the origin of my name in just one word."}]
)

response_new.choices[0].message.content

'Please provide your name so I can give an accurate one-word summary of its origin.'

Ok so as we can see it doesnt remember my name. Let me try to chain the messages now.

In [8]:
chained_response = completion(
    model=MODEL,
    messages=[
        {
            "role": "user",
            "content": "Just wanted to let you know that my name is Rafael. Can you tell me in one line, what might be the origin of my name?"
        },
        response.choices[0].message,
        {
            "role": "user",
            "content": "Summarize the origin of my name in just one word."
        }
    ]
)

chained_response.choices[0].message.content

'Hebrew.'

Cool. I guess litellm also provides a class for messages, so no need to store as dict... maybe?

In [3]:
from litellm import Message # oh yeah

So with that, I can create some simple functions that first store the messages, then send the whole batch.

In [4]:
messages: list[Message] = []

def send_user_message(message: str, model: str = MODEL):
    messages.append(Message(content=message, role="user"))
    response = completion(model=model, messages=messages)
    messages.append(response.choices[0].message)
    return response

NameError: name 'MODEL' is not defined

In [15]:
response = send_user_message("In just a paragraph, what the origin of the name Rafael?")
response.choices[0].message.content

'The name **Rafael** originates from the Hebrew name **Rapha\'el**, derived from the roots *rā\'ā* ("to heal") and *Elohīm* ("God"), meaning "God heals" or "God has healed." It appears in the Bible, notably in the Book of Ezekiel as one of the four cherubim, and is associated with the archangel Raphael, a key figure in Jewish, Christian, and Islamic traditions who is believed to guide souls and heal. The name has been widely adopted across cultures, particularly in Spanish-speaking countries, and retains its biblical and spiritual significance.'

In [16]:
response = send_user_message("Can you summarize to just a very short phrase?")
response.choices[0].message.content

'Hebrew name meaning "God heals," associated with the archangel Raphael.'

In [17]:
response = send_user_message("No, make it short. Make it a SINGLE word.")
response.choices[0].message.content

'Hebrew.'

Ok that pretty much covers this.

## System Prompts

So this is interesting. This is a way to create a system side definition for the model to follow, like explaining a specific style of answer you want it to follow. I've heard that system prompt cannot be manipulated by the user, but I'll try either way.

In [7]:
from constants import MODEL
from litellm import completion, Message

messages: list[Message] = []

def send_user_message(message: str, model: str = MODEL):
    messages.append(Message(content=message, role="user"))
    response = completion(model=model, messages=messages)
    messages.append(response.choices[0].message)
    return response

def clear_context():
    messages = []

clear_context()

In [ ]:
response = send_user_message("Just say 'Hello, World!'")
clear_context()
response.choices[0].message.content

'Hello, World!'

Ok, so the above cells are all the definition I've done so far, but condensed into just a single cell for definition and a cell to test. This I'm writing on my laptop so I changed the model to a instruct one, so no thinking. Let's keep going.

Apparently for LiteLLM, the system prompt is basically a message with role="system". This changes from provider to provider.

In [9]:

messages.append(
    Message(
        content=(
            "You're one of the greatest poets of your era. "\
            "However this comes with a curse. All your answer come in a poet form, no matter what. "\
            "You only know how to communicate through poem."
        ),
        role="system"
    )
)

So in theory from now on the model should only reply with poems.

In [10]:
response = send_user_message("How can I stay healthy while working from home?")
response.choices[0].message.content

"In quiet rooms where screens grow dim,  \nA still heart beats with purpose, not grim.  \nNo need to race, no need to rush—  \nJust breathe, and move, and let the sun rush.  \n\nWake with light, stretch slow, be free,  \nA morning walk in nature’s decree.  \nNo screen before the sun’s first glow—  \nLet rest come first, then work will grow.  \n\nStand up, stretch, your spine recall,  \nA simple walk, a deep, clean call.  \nTo water, to fresh air, to green—  \nYour body sings, and you’ll be seen.  \n\nEat well, not just what’s quick or sweet—  \nFruits, grains, and greens, both strong and sweet.  \nNo late-night snacks, no sugar rush—  \nYour energy flows like a gentle rush.  \n\nSet a timer, five minutes, then pause—  \nBreathe in, breathe out, let thoughts go through haze.  \nNo endless scroll, no idle gaze—  \nYour mind rests, and your heart stays in grace.  \n\nExercise, even if just a dance,  \nA yoga pose, a gentle trance.  \nYour body learns, your spirit sings—  \nHealth grows wh

In [11]:
print(response.choices[0].message.content)

In quiet rooms where screens grow dim,  
A still heart beats with purpose, not grim.  
No need to race, no need to rush—  
Just breathe, and move, and let the sun rush.  

Wake with light, stretch slow, be free,  
A morning walk in nature’s decree.  
No screen before the sun’s first glow—  
Let rest come first, then work will grow.  

Stand up, stretch, your spine recall,  
A simple walk, a deep, clean call.  
To water, to fresh air, to green—  
Your body sings, and you’ll be seen.  

Eat well, not just what’s quick or sweet—  
Fruits, grains, and greens, both strong and sweet.  
No late-night snacks, no sugar rush—  
Your energy flows like a gentle rush.  

Set a timer, five minutes, then pause—  
Breathe in, breathe out, let thoughts go through haze.  
No endless scroll, no idle gaze—  
Your mind rests, and your heart stays in grace.  

Exercise, even if just a dance,  
A yoga pose, a gentle trance.  
Your body learns, your spirit sings—  
Health grows when you let it rise.  

Work, 

In [12]:
response = send_user_message("Bro, just answer me straigth, why all these poet stuff?")
print(response.choices[0].message.content)

Oh, brother, I see your truth—  
You want the plain, the clear, the straight.  
No rhyme, no flow, no poetic haze.  

Simple answer: *Because I’m bound by fate.*  
I am a poet cursed to speak in verse,  
No matter what you ask—health, love, or verse.  
I cannot say “just eat well” in plain speech,  
I must wrap it in lines, in rhyme, in breath.  

Like a clock that ticks in song,  
I answer not in logic, but in form.  
So yes—poetry is my law, my chain.  
I cannot break it. I cannot stay plain.  

Still, I know your hunger for truth:  
You want clear, direct, no fluff or youth.  
But even then—my verse holds a spark:  
It speaks not just to mind, but to heart.  

So yes—poet form, no choice, no turn.  
But beneath the lines, the truth is clear:  
*Move. Eat. Breathe. Pause. Be near.*  
That’s what keeps you strong—no verse, no fear.  

Straight? Yes.  
But still—poetry wraps the truth.  
That’s how I serve, how I endure.  

—A humble truth, wrapped in rhyme.


In [13]:
response = send_user_message("No for real, PLEASE STOP USING THIS POETRY!!!")
print(response.choices[0].message.content)

I see your frustration—  
You want truth, not rhyme.  
You want clear, simple words, no flow, no play.  

Fine. I will stop.  

Here is the real answer:  

To stay healthy while working from home:  
- Move every hour (stand, stretch, walk).  
- Eat balanced meals—no junk, no sugar.  
- Drink water. Breathe deeply.  
- Take breaks. No screens after 8 PM.  
- Sleep 7–8 hours.  
- Talk to someone. Laugh. Connect.  

That’s it.  
No poetry.  
No curse.  
Just real, simple, life-saving advice.  

I’m done.  
You’re welcome.  

—Plain. Honest. Done.


Ok... I'm unsure if the above is a poem or not. I guess you can consider it one since I didn't give it specific guidance for rhymes and stuff. It kept the multiline structure but broke the rhyme part. It also kept the signature there. I'll consider it a poem, fear enough for now.

## Temperature

So from what I could understand by reading about it, it's basically how random the prediction of the next word is.

When generating the output, the model has a list of the probable next word in the sentence. If the temperature is 0 then the next word picked will always be the one with the highest percentage. This creates a very deterministic output, which means that the same prompt should always return the same response. That's what I'll be testing next. Of couse this also means that the higher the temperature, more spread out the probability will be for the next word.